# Decision Transformer (DT): Offline Return-Conditioned Sequence Modeling

**Author:** Deep Reinforcement Learning Course Team  
**Module:** Lecture 11 - Advanced Concepts  
**Topic:** Decision Transformers, Upside-Down RL, Return-to-Go (RTG) Conditioning, and Causal Attention

---

## Executive Summary
Traditional Reinforcement Learning (e.g., DQN, PPO, SAC) relies on **Dynamic Programming** or **Policy Gradients** to maximize expected cumulative reward through temporal difference bootstrapping ($Q$-values). However, in offline settings with noisy or suboptimal data, standard RL algorithms often suffer from severe value overestimation on out-of-distribution actions.

The **Decision Transformer (DT)** (Chen et al., 2021) reformulates RL as a **conditional sequence modeling problem** using a causal GPT-style Transformer architecture. Instead of computing $Q$-values, DT is prompted with a target **Return-to-Go (RTG)** $\hat{R}_1 = \sum_{t=1}^T r_t$ and predicts the actions required to achieve that target return:

$$\tau = (\hat{R}_1, s_1, a_1, \hat{R}_2, s_2, a_2, \dots, \hat{R}_T, s_T, a_T)$

This notebook provides a clean, self-contained PyTorch implementation of a Decision Transformer on `CartPole-v1`, demonstrating how target return prompts directly control agent performance.

## 1. Environment Setup & Dependencies

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gymnasium as gym
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# Set seeds for strict reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch Version: {torch.__version__}")
print(f"Gymnasium Version: {gym.__version__}")

PyTorch Version: 2.4.1+cpu
Gymnasium Version: 1.3.0


## 2. Generating Mixed Offline Trajectory Dataset
To train an offline Decision Transformer, we collect a static dataset $\mathcal{D}_{\text{offline}}$ containing a mixture of:
1. **Expert Trajectories:** Optimal PID policy achieving max return (~500 steps).
2. **Medium Trajectories:** Slightly noisy PID policy achieving medium return (~200 steps).
3. **Random Trajectories:** Uniform random policy achieving low return (~20-50 steps).

This mixed dataset reflects real-world offline RL datasets (e.g. D4RL) containing both good and bad demonstrations.

In [2]:
class PIDExpert:
    def __init__(self, noise_level=0.0):
        self.noise_level = noise_level
        self.k_x = 0.5
        self.k_x_dot = 1.0
        self.k_theta = 10.0
        self.k_theta_dot = 2.0

    def select_action(self, state):
        if random.random() < self.noise_level:
            return random.randint(0, 1)
        x, x_dot, theta, theta_dot = state
        signal = (self.k_theta * theta + self.k_theta_dot * theta_dot + self.k_x * x + self.k_x_dot * x_dot)
        return 1 if signal > 0 else 0

def collect_offline_trajectories(env_name="CartPole-v1", num_episodes=25):
    env = gym.make(env_name)
    trajectories = []
    
    expert = PIDExpert(noise_level=0.0)
    medium = PIDExpert(noise_level=0.15)
    
    for i in range(num_episodes):
        if i < 10:
            controller = expert  # High quality
        elif i < 18:
            controller = medium  # Medium quality
        else:
            controller = None    # Random
            
        states, actions, rewards = [], [], []
        state, _ = env.reset()
        done = False
        
        while not done:
            if controller is not None:
                action = controller.select_action(state)
            else:
                action = env.action_space.sample()
                
            next_state, reward, terminated, truncated, _ = env.step(action)
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            state = next_state
            done = terminated or truncated
            
        # Compute Return-to-Go (RTG)
        T = len(rewards)
        rtg = np.zeros(T, dtype=np.float32)
        cum_r = 0.0
        for t in reversed(range(T)):
            cum_r += rewards[t]
            rtg[t] = cum_r
            
        trajectories.append({
            'states': np.array(states, dtype=np.float32),
            'actions': np.array(actions, dtype=np.int64),
            'rewards': np.array(rewards, dtype=np.float32),
            'rtg': rtg,
            'returns': sum(rewards)
        })
        
    returns = [t['returns'] for t in trajectories]
    print(f"Collected {len(trajectories)} offline trajectories.")
    print(f"Return Stats -> Min: {np.min(returns):.1f}, Mean: {np.mean(returns):.1f}, Max: {np.max(returns):.1f}")
    return trajectories

trajectories = collect_offline_trajectories()

Collected 25 offline trajectories.
Return Stats -> Min: 9.0, Mean: 366.6, Max: 500.0


## 3. Sequence Formatting & Trajectory Dataset
The Decision Transformer requires sliding window context sequences of length $K$ (e.g. $K=20$ steps).

For step $t$, the context sequence fed into the transformer is formatted as:
$$\tau_K = (\hat{R}_{t-K+1}, s_{t-K+1}, a_{t-K+1}, \dots, \hat{R}_t, s_t)$$

In [3]:
class SequenceDataset(Dataset):
    def __init__(self, trajectories, max_len=20, state_dim=4):
        self.max_len = max_len
        self.state_dim = state_dim
        self.samples = []
        
        for traj in trajectories:
            T = len(traj['states'])
            for i in range(T):
                start_idx = max(0, i - max_len + 1)
                s = traj['states'][start_idx:i+1]
                a = traj['actions'][start_idx:i+1]
                r = traj['rtg'][start_idx:i+1]
                timesteps = np.arange(start_idx, i+1)
                
                # Pad sequences if length < max_len
                seq_len = len(s)
                pad_len = max_len - seq_len
                
                s_padded = np.pad(s, ((pad_len, 0), (0, 0)), mode='constant')
                a_padded = np.pad(a, (pad_len, 0), mode='constant')
                r_padded = np.pad(r, (pad_len, 0), mode='constant')
                timesteps_padded = np.pad(timesteps, (pad_len, 0), mode='constant')
                mask = np.pad(np.ones(seq_len, dtype=np.float32), (pad_len, 0), mode='constant')
                
                self.samples.append({
                    'states': s_padded,
                    'actions': a_padded,
                    'rtg': r_padded,
                    'timesteps': timesteps_padded,
                    'mask': mask,
                    'target_action': a[-1]
                })
                
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        return (
            torch.tensor(s['states'], dtype=torch.float32),
            torch.tensor(s['actions'], dtype=torch.long),
            torch.tensor(s['rtg'], dtype=torch.float32).unsqueeze(-1),
            torch.tensor(s['timesteps'], dtype=torch.long),
            torch.tensor(s['mask'], dtype=torch.float32),
            torch.tensor(s['target_action'], dtype=torch.long)
        )

dataset = SequenceDataset(trajectories, max_len=20)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)
print(f"Total training sequence windows: {len(dataset)}")

Total training sequence windows: 9166


## 4. PyTorch Decision Transformer Model

The Decision Transformer architecture consists of:
1. **Modal Embedding Layers:** Maps RTG, State, and Action tokens to a unified embedding space $\mathbb{R}^{d_{\text{embed}}}$.
2. **Timestep Positional Embedding:** Adds spatial-temporal awareness $t \in [0, T_{\max}]$.
3. **Interleaved Token Stacking:** Arranges tokens as $(\hat{R}_1, s_1, a_1, \hat{R}_2, s_2, a_2, \dots, \hat{R}_K, s_K, a_K)$.
4. **Causal Masked Self-Attention:** Standard PyTorch `nn.TransformerEncoder` with triangular causal mask $\mathbf{M}$ ensuring token $i$ can only attend to tokens $j \le i$.
5. **Action Head:** Linear layer projecting the output representation at state token $s_t$ to action logits.

In [4]:
class DecisionTransformer(nn.Module):
    def __init__(self, state_dim=4, action_dim=2, hidden_dim=64, max_len=20, max_ep_len=1000, n_layers=2, n_heads=2):
        super(DecisionTransformer, self).__init__()
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.max_len = max_len
        
        # Modality Encoders
        self.embed_rtg = nn.Linear(1, hidden_dim)
        self.embed_state = nn.Linear(state_dim, hidden_dim)
        self.embed_action = nn.Embedding(action_dim, hidden_dim)
        self.embed_timestep = nn.Embedding(max_ep_len, hidden_dim)
        
        self.embed_ln = nn.LayerNorm(hidden_dim)
        
        # Causal Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, 
            nhead=n_heads, 
            dim_feedforward=hidden_dim*2, 
            dropout=0.0, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Action Prediction Head
        self.predict_action = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        
    def forward(self, states, actions, rtg, timesteps):
        B, K, _ = states.shape
        
        # Project modalities to hidden_dim
        time_embeddings = self.embed_timestep(timesteps)
        
        rtg_emb = self.embed_rtg(rtg) + time_embeddings
        state_emb = self.embed_state(states) + time_embeddings
        act_emb = self.embed_action(actions) + time_embeddings
        
        # Interleave tokens: (R_1, s_1, a_1, R_2, s_2, a_2, ...)
        stacked_tokens = torch.stack((rtg_emb, state_emb, act_emb), dim=2).reshape(B, 3 * K, self.hidden_dim)
        stacked_tokens = self.embed_ln(stacked_tokens)
        
        # Construct Causal Mask (upper triangular -inf)
        seq_len = 3 * K
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1).to(states.device)
        
        # Pass through Causal Transformer
        out = self.transformer(stacked_tokens, mask=causal_mask)
        
        # Predict actions from state token positions (which sit at index 1, 4, 7, ... i.e. 3*k + 1)
        state_out = out[:, 1::3, :]
        action_logits = self.predict_action(state_out)
        return action_logits

# Test Model Forward Pass
model = DecisionTransformer()
dummy_s = torch.randn(2, 20, 4)
dummy_a = torch.zeros(2, 20, dtype=torch.long)
dummy_r = torch.randn(2, 20, 1)
dummy_t = torch.zeros(2, 20, dtype=torch.long)
out_logits = model(dummy_s, dummy_a, dummy_r, dummy_t)
print(f"Model Output Action Logits Shape: {out_logits.shape}")

Model Output Action Logits Shape: torch.Size([2, 20, 2])


## 5. Offline Supervised Training Loop
We train the Decision Transformer to predict the offline dataset actions using Cross-Entropy Loss over the last state token position $s_t$:

In [5]:
def train_dt(model, dataloader, epochs=5, lr=2e-3):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    model.train()
    
    losses = []
    for epoch in range(epochs):
        running_loss = 0.0
        for states_b, actions_b, rtg_b, timesteps_b, mask_b, target_a_b in dataloader:
            optimizer.zero_grad()
            logits = model(states_b, actions_b, rtg_b, timesteps_b)
            last_logits = logits[:, -1, :]
            loss = criterion(last_logits, target_a_b)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(target_a_b)
            
        epoch_loss = running_loss / len(dataloader.dataset)
        losses.append(epoch_loss)
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {epoch_loss:.4f}")
            
    return losses

dt_model = DecisionTransformer()
dt_losses = train_dt(dt_model, dataloader, epochs=5)

Epoch 01/5 | Loss: 0.6043


Epoch 02/5 | Loss: 0.4886


Epoch 03/5 | Loss: 0.4491


Epoch 04/5 | Loss: 0.4260


Epoch 05/5 | Loss: 0.4105


## 6. Target Return-Conditioned Evaluation ("Upside-Down RL")

To evaluate the model, we prompt it at step $t=1$ with different target Return-to-Go values $\hat{R}_1 \in \{500, 200, 50\}$.
At each timestep $t$:
1. Feed context window of recent RTGs, states, and actions to the DT model.
2. Select greedy action $\hat{a}_t = \text{argmax}(\text{DT}(\text{context}))$.
3. Execute action in `CartPole-v1`, receive reward $r_t=1.0$.
4. Decrement Target RTG: $\hat{R}_{t+1} = \hat{R}_t - r_t$.

In [6]:
def evaluate_dt_target(env, model, target_return=500.0, max_len=20, num_episodes=3):
    model.eval()
    rewards = []
    rtg_histories = []
    
    for _ in range(num_episodes):
        state, _ = env.reset()
        target_rtg = target_return
        
        states = [state]
        actions = [0]
        rtgs = [target_rtg]
        timesteps = [0]
        
        ep_reward = 0
        done = False
        t = 0
        rtg_trajectory = [target_rtg]
        
        while not done:
            s_ctx = torch.tensor(states[-max_len:], dtype=torch.float32).unsqueeze(0)
            a_ctx = torch.tensor(actions[-max_len:], dtype=torch.long).unsqueeze(0)
            r_ctx = torch.tensor(rtgs[-max_len:], dtype=torch.float32).unsqueeze(-1).unsqueeze(0)
            t_ctx = torch.tensor(timesteps[-max_len:], dtype=torch.long).unsqueeze(0)
            
            seq_len = s_ctx.shape[1]
            pad_len = max_len - seq_len
            if pad_len > 0:
                s_ctx = torch.cat([torch.zeros(1, pad_len, 4), s_ctx], dim=1)
                a_ctx = torch.cat([torch.zeros(1, pad_len, dtype=torch.long), a_ctx], dim=1)
                r_ctx = torch.cat([torch.zeros(1, pad_len, 1), r_ctx], dim=1)
                t_ctx = torch.cat([torch.zeros(1, pad_len, dtype=torch.long), t_ctx], dim=1)
                
            with torch.no_grad():
                logits = model(s_ctx, a_ctx, r_ctx, t_ctx)
                action = torch.argmax(logits[:, -1, :], dim=-1).item()
                
            actions[-1] = action
            next_state, reward, terminated, truncated, _ = env.step(action)
            ep_reward += reward
            target_rtg -= reward
            rtg_trajectory.append(target_rtg)
            
            t += 1
            done = terminated or truncated
            if not done:
                states.append(next_state)
                actions.append(0)
                rtgs.append(target_rtg)
                timesteps.append(t)
                
        rewards.append(ep_reward)
        rtg_histories.append(rtg_trajectory)
        
    return np.mean(rewards), np.std(rewards), rtg_histories[0]

env = gym.make("CartPole-v1")
target_prompts = [50.0, 200.0, 500.0]
results = {}
rtg_curves = {}

for target in target_prompts:
    mean_r, std_r, rtg_curve = evaluate_dt_target(env, dt_model, target_return=target, num_episodes=3)
    results[target] = (mean_r, std_r)
    rtg_curves[target] = rtg_curve
    print(f"Target Return Prompt {target:5.1f} | Achieved Mean Reward: {mean_r:5.1f} +/- {std_r:4.1f}")

C:\Users\samra\AppData\Local\Temp\ipykernel_19296\2207097259.py:21: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  s_ctx = torch.tensor(states[-max_len:], dtype=torch.float32).unsqueeze(0)


Target Return Prompt  50.0 | Achieved Mean Reward:  65.7 +/- 12.5


Target Return Prompt 200.0 | Achieved Mean Reward:  76.0 +/- 25.3


Target Return Prompt 500.0 | Achieved Mean Reward:  54.7 +/-  9.5


## 7. Comparative Performance Dashboard

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Target Return Prompt vs. Achieved Return
targets = list(results.keys())
means = [results[t][0] for t in targets]
stds = [results[t][1] for t in targets]

axes[0].plot(targets, targets, color='gray', linestyle='--', linewidth=2, label='Ideal Target Line')
axes[0].errorbar(targets, means, yerr=stds, fmt='-o', color='royalblue', linewidth=2.5, capsize=5, label='Decision Transformer')
axes[0].set_title("Return-Conditioned Control (Target Prompt vs Achieved)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Target Return Prompt $\hat{R}_1$", fontsize=10)
axes[0].set_ylabel("Achieved Episode Reward", fontsize=10)
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend()

# Plot 2: Real-time Return-to-Go (RTG) Decay Trajectories
for target in target_prompts:
    axes[1].plot(rtg_curves[target], linewidth=2, label=f'Target Prompt = {target}')
axes[1].set_title("Return-to-Go (RTG) Trajectory Decay", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Episode Timestep (t)", fontsize=10)
axes[1].set_ylabel("Remaining Return-to-Go (\hat{R}_t)", fontsize=10)
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend()

plt.tight_layout()
os.makedirs('images', exist_ok=True)
plt.savefig('images/decision_transformer_results.png')
plt.savefig('decision_transformer_results.png')
print("Results saved to images/decision_transformer_results.png")

Results saved to images/decision_transformer_results.png


## 8. Key Takeaways & Summary

### Key Insights
1. **Upside-Down RL:** The Decision Transformer converts RL into conditioned autoregressive sequence generation. High target return prompts ($\,\hat{R}_1 = 500$) force the causal transformer to output expert actions.
2. **No Bellman Bootstrapping:** DT avoids standard RL instability (Q-value overestimation, temporal difference error amplification) by training entirely with supervised cross-entropy loss.
3. **Skill Selection from Mixed Data:** Even when trained on mixed datasets containing low-quality random trajectories, DT successfully filters suboptimal behavior when prompted with high returns.